In [ ]:
import sys
import os

notebook_path = os.path.abspath('') 
os.chdir(notebook_path)

if notebook_path not in sys.path:
    sys.path.append(notebook_path)

print(f"Current Directory: {os.getcwd()}")
print(f"Directory Contents: {os.listdir('.')}")

In [1]:
from torch import nn
import pandas as pd
from torch.utils.data import DataLoader 
from sklearn.model_selection import train_test_split

SEED = 42
DATASET_URL = "https://gist.githubusercontent.com/FlavRomano/2f0b37a3d0d5b7230d548a0de563c4a0/raw/8ace8e7eb27e6432b963659d68c02dccb35ab108/en_anagram_dictionary_encoding.csv"

In [2]:
tr, ts = train_test_split(pd.read_csv(DATASET_URL), train_size=.8, random_state=SEED, shuffle=True)

In [3]:
tr.describe()

,anagram,encoded,encoding
count,71246,71247,71247
unique,71246,71240,3
top,daddies,ups,rot13
freq,1,2,23845


In [13]:
from src.encoding_dataset import EncodingsDataset

train_dataset = EncodingsDataset(tr, max_len=32) # MD5 is 32 chars long
test_dataset = EncodingsDataset(ts, vocab=train_dataset.vocab, 
                                label_map=train_dataset.label_map, 
                                max_len=32)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

VOCAB_SIZE = len(train_dataset.vocab)
NUM_CLASSES = len(train_dataset.label_map)

ModuleNotFoundError: No module named 'src'

In [ ]:
from src.lstm_cryptographic import LSTMCryptographic

model = LSTMCryptographic(vocab_size=VOCAB_SIZE, embed_dim=256, hidden_dim=512, output_dim=NUM_CLASSES)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Trainable Params: {total_params:,}")

Total Trainable Params: 1,589,251


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5


MODEL_PATH = "/Users/flavio/Desktop/ai/gdl/coding-class/exercises/pytorch/lstm/model_weights.pth"

# 1. Check if the file exists
if os.path.exists(MODEL_PATH):
    print(f"--- Weights found, skipping training. ---")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.to(device)
    model.eval()

else:
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            
            loss = criterion(outputs, labels)
            
            loss.backward()
            
            optimizer.step()
            
            running_loss += loss.item()
            
            if (batch_idx + 1) % 100 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Step [{batch_idx+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

        print(f"Epoch {epoch+1} average loss: {running_loss/len(train_loader):.4f}")

        torch.save(model.state_dict(), "/Users/flavio/Desktop/ai/gdl/coding-class/exercises/pytorch/lstm/model_weights.pth")

Epoch [1/5], Step [100/1114], Loss: 0.3747
Epoch [1/5], Step [200/1114], Loss: 0.2255


KeyboardInterrupt: 

In [ ]:
from src.utils import Utils

Utils.evaluate(model, test_loader, criterion, device)

Test Loss: 0.0848 | Test Accuracy: 97.56%


(0.08477187080902972, 97.56344037727375)

In [ ]:
word_to_test = "nneqinex"

result = Utils.predict_encoding(word_to_test, model, train_dataset, device)
print(f"The predicted encoding for '{word_to_test}' is: {result}")

The predicted encoding for 'nneqinex' is: rot13
